# 03 - Veri Ön İşleme ve Ortak Pipeline

**Sorumlu:** Kişi 3

**Ödevdeki karşılığı:** 2. Veri Ön İşleme (D1, D2, D3, D6, D7 Görevleri)

**Amaç:** Bu not defterinde, Bank Marketing veri seti üzerinde ham tekrar kayıtların temizlenmesi, eksik ve `unknown` değerlerin yönetimi, `pdays`/`previous` ayrımının doğru kurulması, sayısal değişkenlerin ölçeklendirilmesi, kategorik değişkenlerin kodlanması, ortak `ColumnTransformer` ve `Pipeline` mimarisinin inşası ile veri sızıntısını önleyecek **%70 Eğitim / %15 Doğrulama (Validation) / %15 Test** tabakalı (stratified) veri ayrımı gerçekleştirilmektedir.

## 1. Çalışma Ortamı ve Kütüphanelerin Yüklenmesi
> **Açıklama:** Bu hücrede veri analizi kütüphaneleri (pandas, numpy) ile tüm modellerin ortak kullandığı ön işleme ve veri bölme modülleri (`src/ortak.py`, `src/preprocessing.py`) içe aktarılır.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Çalışma dizinini kontrol et ve proje kök dizinine sabitle
mevcut_dizin = Path.cwd()
if mevcut_dizin.name == "notebooks":
    os.chdir(mevcut_dizin.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Ortak veri yükleme, bölme, ön işleme ve doğrulama fonksiyonları
from src.ortak import (
    _bul_veri_yolu, veri_yukle, veriyi_bol, on_isleyici, ortak_cv, RANDOM_STATE
)
from src.preprocessing import (
    numeric_features, mode_features, unknown_features, other_cat_features
)

print("Aktif Çalışma Dizini:", Path.cwd())
print("Ortak modüller (src.ortak ve src.preprocessing) başarıyla yüklendi!")

Aktif Çalışma Dizini: c:\Users\nusayba\OneDrive\Masaüstü\makineprojesi\ml-proje
Ortak modüller (src.ortak ve src.preprocessing) başarıyla yüklendi!


## 2. Veri Setinin Ortak Fonksiyonla Okunması ve Boyut Kontrolü
> **Açıklama:** Bank Marketing verisi `src.ortak.veri_yukle()` ortak fonksiyonu aracılığıyla noktalı virgül (`;`) ayırıcısıyla okunur. Veri boyutu, satır ve sütun sayıları kontrol edilir.

In [2]:
# Ham verideki gerçek tekrarları duration sütunu henüz mevcutken raporlayalım
ham_df = pd.read_csv(_bul_veri_yolu(), sep=";")
tekrar_mask = ham_df.duplicated(keep="first")
ham_tekrar_sayisi = int(tekrar_mask.sum())
silinen_y_dagilim = ham_df.loc[tekrar_mask, "y"].value_counts().to_dict()

# Veri setini ortak veri_yukle() fonksiyonu ile güvenli şekilde okuyalım
df = veri_yukle()

print("VERİ BOYUTU VE TEKRAR KONTROLÜ:")
print(f"Ham Satır Sayısı (CSV)                : {len(ham_df)} (y='yes' oranı: %{(ham_df['y']=='yes').mean()*100:.4f})")
print(f"Silinen Gerçek Tekrar (duration dahil): {ham_tekrar_sayisi} (Hedef dağılımı: {silinen_y_dagilim})")
print(f"veri_yukle() Sonrası Boyut            : {df.shape[0]} satır, {df.shape[1]} sütun (y=1 oranı: %{df['y'].mean()*100:.4f})")
print("\nİlk 5 Satır:")
display(df.head(5))

VERİ BOYUTU VE TEKRAR KONTROLÜ:
Ham Satır Sayısı (CSV)                : 41188 (y='yes' oranı: %11.2654)
Silinen Gerçek Tekrar (duration dahil): 12 (Hedef dağılımı: {'no': 11, 'yes': 1})
veri_yukle() Sonrası Boyut            : 41176 satır, 23 sütun (y=1 oranı: %11.2663)

İlk 5 Satır:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,daha_once_aranmis,pdays_kayitli
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0


**Yorum: Veri Seti Yapısı, Tekrar Temizliği ve Ortak Yükleme Adımları:**
Bank Marketing ham veri seti **41.188 satır** ve 21 sütundan oluşmaktadır. Ortak `veri_yukle()` fonksiyonu tüm modellerde tutarlılık sağlamak için şu deterministik (veri öğrenmeyen) hazırlıkları uygular:
- CSV okunduktan hemen sonra (`duration` sütunu henüz mevcutken) tüm 21 sütunu birebir aynı olan **12 gerçek tekrar kaydı** (`11 no`, `1 yes`) siler ve veri setini **41.176** tekil kayda indirir. Hedef sınıf oranı `%11.2654`'ten `%11.2663`'e değişir (sınıf dengesi bozulmaz) ve aynı kaydın hem eğitim hem doğrulama/test kümesine düşmesi engellenir (`duration` çıkarıldıktan sonra tekrar silme yapılmaz).
- Düşük oranlı `unknown` değerlerini (`job`, `marital`, `education`, `housing`, `loan`) `NaN` olarak işaretler; `default` sütunundaki `unknown` kategorisini korur.
- `daha_once_aranmis = (previous > 0)` (müşteri önceki kampanyalarda en az bir kez aranmış mı?) ve `pdays_kayitli = (pdays != 999)` (son aramadan bu yana geçen gün sayısı kayıtlı mı?) özelliklerini ayrı ayrı türetir; gerçek bir gün sayısı olmayan `pdays = 999` özel kodunu ise ortak `Pipeline` içindeki `SimpleImputer(strategy='median')` tarafından yalnızca eğitim verisinden öğrenilerek doldurulmak üzere `NaN` yapar.
- Hedef sütun olan `y` değişkenini `yes -> 1`, `no -> 0` şeklinde ikili formata çevirir.

## D1. Eksik, Bilinmeyen (Unknown) ve Tutarsız Değerlerin Analizi
> **Açıklama:** Veri setinde standart `NaN` değerleri ile gizli eksik değer olan `unknown` kayıtları tespit edilir. Sayısal sütunlarda ise `pdays = 999` özel kodu ve IQR tabanlı aykırı değerler incelenir.

In [3]:
# 1. NaN değerlerinin kontrolü (düşük oranlı unknown'lar + pdays=999'dan dönüşen NaN'lar)
nan_sayilari = df.isnull().sum()

# 2. Ayrı kategori olarak korunan 'unknown' değerlerinin tespiti (default sütunu)
unknown_sayilari = (df == "unknown").sum()

eksik_tablosu = pd.DataFrame({
    "NaN Sayısı": nan_sayilari,
    "NaN Oranı (%)": (nan_sayilari / len(df) * 100).round(2),
    "Korunan Unknown Sayısı": unknown_sayilari,
    "Korunan Unknown Oranı (%)": (unknown_sayilari / len(df) * 100).round(2)
})

eksik_tablosu = eksik_tablosu[
    (eksik_tablosu["NaN Sayısı"] > 0) | (eksik_tablosu["Korunan Unknown Sayısı"] > 0)
]
print("Eksik (NaN) ve Korunan Unknown Değer Dağılımı:")
display(eksik_tablosu)

Eksik (NaN) ve Korunan Unknown Değer Dağılımı:


,NaN Sayısı,NaN Oranı (%),Korunan Unknown Sayısı,Korunan Unknown Oranı (%)
job,330,0.80,0,0.00
marital,80,0.19,0,0.00
education,1730,4.20,0,0.00
default,0,0.00,8596,20.88
housing,990,2.40,0,0.00
loan,990,2.40,0,0.00
pdays,39661,96.32,0,0.00


**Yorum: Eksik ve Unknown Değer Yönetim Stratejimiz:**
1. Ham veri setinde standart `NaN` bulunmamakta, ancak 6 farklı kategorik değişkende `unknown` şeklinde kodlanmış gizli eksik veriler yer almaktadır.
2. **`job` (330 adet - %0.80), `marital` (80 adet - %0.19), `education` (1.730 adet - %4.20), `housing` (990 adet - %2.40), `loan` (990 adet - %2.40):** 12 gerçek tekrar silindikten sonra bu sütunlardaki `unknown` oranları %5'in altındadır. Satırları silmek veri kaybına (özellikle zaten az olan pozitif `yes` sınıfının kaybına) yol açacağı için, `prepare_unknown_values()` ile `NaN`'a dönüştürülmüş olup ortak `ColumnTransformer` içinde yalnızca eğitim verisinden öğrenilen en sık değerle (**mode / most_frequent**) doldurulacaktır.
3. **`default` (8.596 adet - %20.88):** Kredi temerrüdü sütununda `unknown` oranı %20.88'dir. Veri setinde temerrüde düşmüş (`yes`) müşteri sayısı yalnızca 3 adettir. `unknown` değerini mode (`no`) ile doldurmak temerrüt risk belirsizliğini tamamen ortadan kaldırır. Müşterinin temerrüt durumunun bilinmemesi kendi başına ayırt edici bir bilgi taşıdığı için bu sütundaki `unknown` değeri **ayrı bir kategori olarak korunmuştur**.

In [4]:
# daha_once_aranmis (previous > 0) ve pdays_kayitli (pdays != 999) ayrımının kontrolü
daha_once_aranmis_sayisi = int((df["daha_once_aranmis"] == 1).sum())
pdays_kayitli_sayisi = int((df["pdays_kayitli"] == 1).sum())
aranmis_ama_pdays_kayitsiz = int(((df["daha_once_aranmis"] == 1) & (df["pdays_kayitli"] == 0)).sum())
print(f"Daha önce aranmış müşteri (previous > 0)          : {daha_once_aranmis_sayisi} (%{daha_once_aranmis_sayisi/len(df)*100:.2f})")
print(f"pdays bilgisi kayıtlı müşteri (ham pdays != 999)  : {pdays_kayitli_sayisi} (%{pdays_kayitli_sayisi/len(df)*100:.2f})")
print(f"Daha önce aranmış fakat pdays=999 (kayıtsız) olan : {aranmis_ama_pdays_kayitsiz} (%{aranmis_ama_pdays_kayitsiz/len(df)*100:.2f})")
print(f"pdays NaN (999'dan dönüşen) sayısı                : {int(df['pdays'].isna().sum())} | Kalan 999: {int((df['pdays'] == 999).sum())}")
print(f"Kayıtlı pdays medyanı (tüm kayıtlı değerler)      : {df['pdays'].median():.1f} gün (aralık: {df['pdays'].min():.0f}-{df['pdays'].max():.0f})")

# Grup bazında hedef oranları (daha_once_aranmis x pdays_kayitli x poutcome)
grup_ozet = df.groupby(["daha_once_aranmis", "pdays_kayitli", "poutcome"]).agg(
    Kayit_Sayisi=("y", "count"),
    Abonelik_Orani_Yuzde=("y", lambda s: round(s.mean() * 100, 2))
)
print("\nÖnceki Kampanya Etkileşimi Çapraz Tablosu:")
display(grup_ozet)

sayisal_sutunlar = [col for col in df.select_dtypes(include=["number"]).columns if col != "y"]
aykiri_raporu = []

for col in sayisal_sutunlar:
    seri = df[col].dropna()
    Q1 = seri.quantile(0.25)
    Q3 = seri.quantile(0.75)
    IQR = Q3 - Q1
    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR
    aykiri_sayisi = int(((seri < alt_sinir) | (seri > ust_sinir)).sum())
    aykiri_raporu.append({
        "Sütun": col,
        "Min": seri.min(),
        "Medyan": seri.median(),
        "Maksimum": seri.max(),
        "Aykırı Sayısı": aykiri_sayisi,
        "Aykırı Oranı (%)": round((aykiri_sayisi / len(seri)) * 100, 2)
    })

display(pd.DataFrame(aykiri_raporu))

Daha önce aranmış müşteri (previous > 0)          : 5625 (%13.66)
pdays bilgisi kayıtlı müşteri (ham pdays != 999)  : 1515 (%3.68)
Daha önce aranmış fakat pdays=999 (kayıtsız) olan : 4110 (%9.98)
pdays NaN (999'dan dönüşen) sayısı                : 39661 | Kalan 999: 0
Kayıtlı pdays medyanı (tüm kayıtlı değerler)      : 6.0 gün (aralık: 0-27)

Önceki Kampanya Etkileşimi Çapraz Tablosu:


Kayit_Sayisi  \
daha_once_aranmis pdays_kayitli poutcome                    
0                 0             nonexistent         35551   
1                 0             failure              4110   
                  1             failure               142   
                                success              1373   

                                             Abonelik_Orani_Yuzde  
daha_once_aranmis pdays_kayitli poutcome                           
0                 0             nonexistent                  8.83  
1                 0             failure                     12.94  
                  1             failure                     51.41  
                                success                     65.11

,Sütun,Min,Medyan,Maksimum,Aykırı Sayısı,Aykırı Oranı (%)
0,age,17.000,38.000,98.000,468,1.14
1,duration,0.000,180.000,4918.000,2963,7.20
2,campaign,1.000,2.000,56.000,2406,5.84
3,pdays,0.000,6.000,27.000,82,5.41
4,previous,0.000,0.000,7.000,5625,13.66
5,emp.var.rate,-3.400,1.100,1.400,0,0.00
6,cons.price.idx,92.201,93.749,94.767,0,0.00
7,cons.conf.idx,-50.800,-41.800,-26.900,446,1.08
8,euribor3m,0.634,4.857,5.045,0,0.00
9,nr.employed,4963.600,5191.000,5228.100,0,0.00


**Yorum: Sayısal Sütunlar, `previous` / `pdays` Ayrımı ve `pdays = 999 → NaN + Medyan` Tercihi:**
1. **`daha_once_aranmis` vs `pdays_kayitli`:** Veri setinde `previous > 0` olan **5.625** müşteri (%13.66) varken `pdays != 999` olan müşteri sayısı yalnızca **1.515**'tir (%3.68). Aradaki **4.110 müşteri** daha önce aranmış (`poutcome == 'failure'`) olmasına rağmen `pdays = 999` (kayıtsız) değerindedir. Çapraz tabloda abonelik oranları bu ayrımın gücünü net biçimde göstermektedir: `nonexistent` (%8.83), `failure` + `pdays=999` (%12.94), `failure` + `pdays` kayıtlı (%51.41), `success` + `pdays` kayıtlı (%65.11).
2. **`pdays = 999` Neden `-1` Yerine `NaN + SimpleImputer(median)` Yapıldı?:** `999` gerçek bir gün sayısı değildir. `-1` gibi yapay bir sabit kod kullanıldığında hem sayısal eksende `0 gün` değerinin hemen yanına yapay bir komşu eklenir hem de `pdays` ile `pdays_kayitli` arasında yüksek çoklu doğrusallık (`r = 0.8742`) oluşur. Bunun yerine `999 → NaN` yapılıp `pdays_kayitli` göstergesiyle korunduğunda, `SimpleImputer(strategy='median')` yalnızca eğitim katmanındaki kayıtlı günlerin medyanını (~6.0 gün) öğrenerek doldurur ve `pdays` ile `pdays_kayitli` arasındaki korelasyon `0.0037` düzeyine iner (hem doğrusal/uzaklık modellerinde hem SHAP yorumlarında çok daha temiz ve sızıntısız bir temsil sağlanır).
3. **`duration` (Görüşme Süresi):** 0 ile 4.918 saniye arasında değişmektedir; ancak çağrı tamamlandıktan sonra bilindiği için doğrudan bir **Veri Sızıntısı (Data Leakage)** kaynağıdır ve `veriyi_bol()` aşamasında çıkarılır.
4. **`age` ve `campaign`:** Yüksek değerler gerçek müşteri profilini (örneğin emekli kitleyi) temsil ettiğinden silinmez; medyan doldurucu ve `StandardScaler` ile modellenir.

## D2. Hedef Değişkenin Kodlanması, duration Sütununun Çıkarılması ve Ortak %70 / %15 / %15 Veri Bölme
> **Açıklama:** Hedef sütun `y` (0/1) ve `duration` sütununun çıkarılması ile **%70 Eğitim / %15 Doğrulama (Validation) / %15 Test** tabakalı (stratified) veri ayrımı `src.ortak.veriyi_bol()` fonksiyonuyla tüm modeller için aynı şekilde yapılır.

In [5]:
# Ortak veriyi_bol() fonksiyonu 'y' ve 'duration' sütunlarını ayırır, %70 / %15 / %15 tabakalı bölme yapar
X_train, X_val, X_test, y_train, y_val, y_test = veriyi_bol(df)

# Veri sızıntısı kontrolü: duration sütunu üç kümede de kesinlikle çıkarılmış olmalı
for ad, parca in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    assert "duration" not in parca.columns, f"HATA: duration sütunu {ad} içinde kalmış!"

toplam = len(df)
print(f"Toplam Tekil Örnek Sayısı          : {toplam}")
print(f"Model Girdisi Özellik Sayısı       : {X_train.shape[1]} ('duration' ve 'y' çıkarıldı; 'daha_once_aranmis' ve 'pdays_kayitli' eklendi)")
print(f"Eğitim Seti     (X_train) Boyutu   : {X_train.shape} (%{len(X_train)/toplam*100:.2f})")
print(f"Doğrulama Seti  (X_val)   Boyutu   : {X_val.shape} (%{len(X_val)/toplam*100:.2f})")
print(f"Test Seti       (X_test)  Boyutu   : {X_test.shape} (%{len(X_test)/toplam*100:.2f})")
print("\nHedef Değişken Sınıf Dağılımı (Tüm Tekil Veri):")
print(df["y"].value_counts())
print("\nHedef Değişken Yüzdelik Dağılımı (%):")
print((df["y"].value_counts(normalize=True) * 100).round(2))

Toplam Tekil Örnek Sayısı          : 41176
Model Girdisi Özellik Sayısı       : 21 ('duration' ve 'y' çıkarıldı; 'daha_once_aranmis' ve 'pdays_kayitli' eklendi)
Eğitim Seti     (X_train) Boyutu   : (28823, 21) (%70.00)
Doğrulama Seti  (X_val)   Boyutu   : (6176, 21) (%15.00)
Test Seti       (X_test)  Boyutu   : (6177, 21) (%15.00)

Hedef Değişken Sınıf Dağılımı (Tüm Tekil Veri):
y
0    36537
1     4639
Name: count, dtype: int64

Hedef Değişken Yüzdelik Dağılımı (%):
y
0    88.73
1    11.27
Name: proportion, dtype: float64


**Yorum: Veri Sızıntısı ve Sınıf Dengesizliği:**
1. `duration` sütununun düşürülmesiyle modelin çağrı süresine bakarak gerçek dünyada uygulanamaz yapay bir başarı üretmesi engellenmiştir.
2. Tekrarlar temizlendikten sonra kalan **41.176** kaydın **36.537**'si "Hayır" (**%88.73**) ve **4.639**'u "Evet" (**%11.27**) sınıfındadır. Veri kümesi **yüksek oranda dengesizdir (imbalanced)**. Bu nedenle eğitimde `class_weight='balanced'` veya yalnızca eğitim verisine uygulanan SMOTE gibi yöntemler değerlendirilmeli; başarı ölçümünde `Accuracy` yerine `Recall`, `Precision`, `F1-Score` ve `ROC-AUC` metrikleri öne çıkarılmalıdır.

## D3. Sayısal ve Kategorik Sütunların Ayrılması
> **Açıklama:** Sütunlar veri türlerine ve uygulanacak dönüşümlere göre listelere ayrılır. Hangi modellerin ölçeklendirmeye ihtiyaç duyduğu açıklanır.

In [6]:
# Özellik grupları src/preprocessing.py modülünden ortak olarak alınır (ikinci kez tanımlanmaz)
kategorik_ozellikler = mode_features + unknown_features + other_cat_features

print(f"Sayısal Özellik Sayısı ({len(numeric_features)}):", numeric_features)
print(f"\nMode ile Doldurulacak Kategorikler ({len(mode_features)}):", mode_features)
print(f"Unknown Korunacak Kategorikler ({len(unknown_features)}):", unknown_features)
print(f"Diğer Kategorik Özellikler ({len(other_cat_features)}):", other_cat_features)
print(f"\nToplam Kategorik Özellik Sayısı ({len(kategorik_ozellikler)}):", kategorik_ozellikler)

# Doğrulama: Ortak listedeki tüm sütunlar X_train sütunlarıyla birebir eşleşiyor mu?
assert set(numeric_features + kategorik_ozellikler) == set(X_train.columns), "Eksik ya da fazla sütun var!"
print("\nTüm sütunlar src/preprocessing.py tanımlarıyla birebir eşleştirildi ve doğrulandı.")

Sayısal Özellik Sayısı (11): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'daha_once_aranmis', 'pdays_kayitli']

Mode ile Doldurulacak Kategorikler (5): ['job', 'marital', 'education', 'housing', 'loan']
Unknown Korunacak Kategorikler (1): ['default']
Diğer Kategorik Özellikler (4): ['contact', 'month', 'day_of_week', 'poutcome']

Toplam Kategorik Özellik Sayısı (10): ['job', 'marital', 'education', 'housing', 'loan', 'default', 'contact', 'month', 'day_of_week', 'poutcome']

Tüm sütunlar src/preprocessing.py tanımlarıyla birebir eşleştirildi ve doğrulandı.


**Yorum: Hangi Modeller Ölçeklendirmeye İhtiyaç Duyar?:**
- **StandardScaler ($z = \frac{x - \mu}{\sigma}$):** Sayısal sütunların ortalamasını 0, standart sapmasını 1 yapar.
- **Lojistik Regresyon (BİZİM MODELİMİZ):** Gradyan temelli optimizasyon ve katsayı düzenlileştirmesi ($L_1 / L_2$) kullandığından, farklı ölçeklerdeki değişkenler (örneğin yaş: 17-98, nr.employed: 4963-5228) katsayı cezalandırmasını dengesizleştirir. Bu nedenle **Lojistik Regresyon için ölçeklendirme zorunludur**.
- **KNN ve SVM:** Mesafe tabanlı (Öklid mesafesi) veya marjin temelli çalıştıklarından büyük değerli değişkenler mesafeyi domine eder. Ölçeklendirme kesinlikle şarttır.
- **Ağaç Tabanlı Modeller (Random Forest, XGBoost):** Karar sınırlarını sıralı eşik değerleri (threshold) ile belirlediklerinden ölçeklendirmeye ihtiyaç duymazlar ancak ölçeklendirilmiş veriden de olumsuz etkilenmezler.
- **OneHotEncoder:** Kategorik değişkenlerde yapay bir sıralama (örneğin evli > bekar gibi) oluşturmamak için nominal kategoriler ikili (dummy) sütunlara dönüştürülür.

## D6. Ortak ColumnTransformer ve Ön İşleme Mimarisi
> **Açıklama:** Ön işleme adımları bu defterde ikinci kez tanımlanmaz; tüm ekip üyelerinin kullandığı ortak `ColumnTransformer` yapısı doğrudan `src.ortak.on_isleyici()` fonksiyonundan çağrılır.

In [7]:
# Ortak ColumnTransformer nesnesini src/ortak.py üzerinden çağırıyoruz
preprocessor = on_isleyici()

print("Ortak ColumnTransformer (src.ortak.on_isleyici) başarıyla yüklendi:")
print(preprocessor)

Ortak ColumnTransformer (src.ortak.on_isleyici) başarıyla yüklendi:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'campaign', 'pdays', 'previous',
                                  'emp.var.rate', 'cons.price.idx',
                                  'cons.conf.idx', 'euribor3m', 'nr.employed',
                                  'daha_once_aranmis', 'pdays_kayitli']),
                                ('mode',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequen...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),


**Yorum: Ortak Pipeline ve Güvenlik:**
1. **Tek Merkezli Mimari (`src/preprocessing.py` & `src/ortak.py`):** Sayısal sütunlarda `SimpleImputer(strategy='median') + StandardScaler()`, düşük oranlı `unknown` içeren kategoriklerde `SimpleImputer(strategy='most_frequent') + OneHotEncoder()`, `default` ve diğer kategoriklerde doğrudan `OneHotEncoder()` adımları tüm 5 model için aynı bileşenden gelir.
2. **`handle_unknown='ignore'`:** Test setinde veya gerçek hayat verilerinde daha önce görülmemiş yeni bir kategori gelmesi durumunda kodun çökmesini engeller, o kategoriyi tüm sıfırlar olarak kodlar.
3. **`sparse_output=False`:** Çıktının yoğun (dense numpy array) olmasını sağlayarak Lojistik Regresyon, KNN, SVM, Random Forest ve XGBoost modellerinin tamamıyla sorunsuz çalışmasını garanti eder.

## D7. Eğitim / Doğrulama / Test Ayrımı (%70 - %15 - %15 Stratified) ve Ortak 5-Katlı Doğrulama
> **Açıklama:** `veriyi_bol(df)` ile elde edilen %70 Eğitim, %15 Doğrulama (Validation) ve %15 Test ayrımının boyut ve sınıf oranları ile yalnızca eğitim setinde kullanılacak `ortak_cv()` yapılandırması doğrulanır.

In [8]:
cv = ortak_cv()

print("================ ORTAK %70 / %15 / %15 BÖLME SONUÇLARI ================")
print(f"Eğitim Seti     (X_train): {X_train.shape} | y_train: {y_train.shape} | Oran: %{len(X_train)/len(df)*100:.2f}")
print(f"Doğrulama Seti  (X_val)  : {X_val.shape} | y_val  : {y_val.shape} | Oran: %{len(X_val)/len(df)*100:.2f}")
print(f"Test Seti       (X_test) : {X_test.shape} | y_test : {y_test.shape} | Oran: %{len(X_test)/len(df)*100:.2f}")

dagilim_df = pd.DataFrame({
    "Küme": ["Eğitim (Train)", "Doğrulama (Validation)", "Test (Test)"],
    "Kayıt Sayısı": [len(y_train), len(y_val), len(y_test)],
    "Veri Oranı (%)": [round(len(y_train)/len(df)*100, 2), round(len(y_val)/len(df)*100, 2), round(len(y_test)/len(df)*100, 2)],
    "Sınıf 0 (Hayır) %": [round((y_train==0).mean()*100, 2), round((y_val==0).mean()*100, 2), round((y_test==0).mean()*100, 2)],
    "Sınıf 1 (Evet) %": [round((y_train==1).mean()*100, 2), round((y_val==1).mean()*100, 2), round((y_test==1).mean()*100, 2)]
})
display(dagilim_df)
print("\nOrtak Çapraz Doğrulama (Yalnızca X_train üzerinde kullanılacak):")
print(cv)
print(f"Ortak Rastgelelik Sabiti (RANDOM_STATE): {RANDOM_STATE}")

================ ORTAK %70 / %15 / %15 BÖLME SONUÇLARI ================
Eğitim Seti     (X_train): (28823, 21) | y_train: (28823,) | Oran: %70.00
Doğrulama Seti  (X_val)  : (6176, 21) | y_val  : (6176,) | Oran: %15.00
Test Seti       (X_test) : (6177, 21) | y_test : (6177,) | Oran: %15.00


,Küme,Kayıt Sayısı,Veri Oranı (%),Sınıf 0 (Hayır) %,Sınıf 1 (Evet) %
0,Eğitim (Train),28823,70.0,88.73,11.27
1,Doğrulama (Validation),6176,15.0,88.73,11.27
2,Test (Test),6177,15.0,88.73,11.27



Ortak Çapraz Doğrulama (Yalnızca X_train üzerinde kullanılacak):
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Ortak Rastgelelik Sabiti (RANDOM_STATE): 42


**Yorum: Üçlü Tabakalı (Stratified) Bölmenin ve Veri Sızıntısı Önleminin Önemi:**
- İki aşamalı `stratify` bölme (`test_size=0.30` ardından kalan parçada `test_size=0.50`, her ikisinde `random_state=42`) sayesinde **Eğitim (%70)**, **Doğrulama (%15)** ve **Test (%15)** kümelerinin tamamında `%88.73 / %11.27` sınıf dağılımı korunmuştur.
- **Rollerin Ayrımı:** Eğitim verisi (`X_train`) model eğitimi ve 5-katlı çapraz doğrulama için; Doğrulama verisi (`X_val`) model seçimi ve hiperparametre doğrulaması için; Test verisi (`X_test`) ise yalnızca en sonda tek seferlik nihai değerlendirme için kullanılır.
- **VERİ SIZINTISI ÖNLEMİ:** Ön işleme nesnesi (`preprocessor`) ve varsa `SMOTE` yalnızca `X_train` (veya CV içindeki eğitim katları) üzerinde `fit` edilir; `X_val` ve `X_test` üzerinde yalnızca `transform` uygulanır.

## 3. Ön İşlemenin Doğrulanması ve Fit-Transform Testi
> **Açıklama:** ColumnTransformer eğitim verisi üzerinde `fit_transform`, test verisi üzerinde yalnızca `transform` edilerek mimarinin çalıştığı doğrulanır.

In [9]:
# SADECE eğitim verisiyle fit_transform uyguluyoruz!
X_train_islenmis = preprocessor.fit_transform(X_train)

# Validation ve Test verilerini SADECE transform ediyoruz! (Veri sızıntısı yok)
X_val_islenmis = preprocessor.transform(X_val)
X_test_islenmis = preprocessor.transform(X_test)

# İşlenmiş verilerde hiç NaN kalmadığını doğrula
train_nan = int(np.isnan(X_train_islenmis).sum())
val_nan = int(np.isnan(X_val_islenmis).sum())
test_nan = int(np.isnan(X_test_islenmis).sum())
assert train_nan == 0 and val_nan == 0 and test_nan == 0, "İşlenmiş veride NaN kaldı!"

ozellik_adlari = preprocessor.get_feature_names_out()

print("================ ÖN İŞLEME TEST RAPORU ================")
print(f"Girdi Özellik Sayısı      : {X_train.shape[1]}")
print(f"İşlenmiş Özellik Sayısı   : {X_train_islenmis.shape[1]}")
print(f"X_train_islenmis Boyutu   : {X_train_islenmis.shape}")
print(f"X_val_islenmis Boyutu     : {X_val_islenmis.shape}")
print(f"X_test_islenmis Boyutu    : {X_test_islenmis.shape}")
print(f"Eğitim / Val / Test NaN   : {train_nan} / {val_nan} / {test_nan} (Tümü 0 olmalı)")
print(f"\nİlk 11 Özellik İsmi (Sayısal):\n", list(ozellik_adlari[:11]))
print("\nÖn işleme mimarisi veri sızıntısı olmadan %100 başarıyla doğrulandı!")

================ ÖN İŞLEME TEST RAPORU ================
Girdi Özellik Sayısı      : 21
İşlenmiş Özellik Sayısı   : 59
X_train_islenmis Boyutu   : (28823, 59)
X_val_islenmis Boyutu     : (6176, 59)
X_test_islenmis Boyutu    : (6177, 59)
Eğitim / Val / Test NaN   : 0 / 0 / 0 (Tümü 0 olmalı)

İlk 11 Özellik İsmi (Sayısal):
 ['num__age', 'num__campaign', 'num__pdays', 'num__previous', 'num__emp.var.rate', 'num__cons.price.idx', 'num__cons.conf.idx', 'num__euribor3m', 'num__nr.employed', 'num__daha_once_aranmis', 'num__pdays_kayitli']

Ön işleme mimarisi veri sızıntısı olmadan %100 başarıyla doğrulandı!


**Yorum: Nihai Doğrulama, Veri Sızıntısı Kontrolü ve Ekip Modelleri İçin Çıkarımlar:**
1. `duration` ve `y` çıkarıldıktan sonra kalan 19 ham özellik ile türetilen 2 özellik (`daha_once_aranmis` ve `pdays_kayitli`) olmak üzere toplam **21 girdi özelliği**, `OneHotEncoder` sonrasında **59 sayısal özelliğe** dönüşmüştür.
2. Eğitim, doğrulama ve test kümelerinin hiçbirinde `NaN` kalmamıştır (`0 / 0 / 0`). Sayısal özellikler medyan ile doldurulup standartlaştırılmış, kategorik özellikler ise `handle_unknown='ignore'` güvencesiyle kodlanmıştır.
3. **Veri Sızıntısı Yoktur:** Ham tekrar temizliği (`duration` varken) ve `previous`/`pdays` dönüşümleri veriden istatistik öğrenmez. Tüm imputer, scaler ve encoder parametreleri yalnızca `X_train` üzerinde (`fit_transform`) öğrenilmiş, `X_val` ve `X_test` üzerinde yalnızca `transform` uygulanmıştır.
4. Bu ortak omurga (`src/ortak.py`, `src/preprocessing.py`, `src/pipeline.py`) projedeki beş modelin tamamı için tek standart olarak hazırdır.